In [21]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import torchvision
from torchvision.datasets import CIFAR10
import torchvision.transforms as transforms

In [22]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train = CIFAR10(root="./data", download=False, train=True, transform=transform)
test = CIFAR10(root="./data", train=False, download=False, transform=transform)

In [23]:
trainloader = DataLoader(train, batch_size=64, shuffle=True)
testloader = DataLoader(test, batch_size=64)

In [24]:
class CNN(nn.Module):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.conv_layers = nn.Sequential(
            # Starting image size 32 * 32 * 3
            # Max pool layer will descrese the width and height 
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # 16(w), 16(h), 32(out_channels)

            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # 8(w), 8(h), 64(out_channels)

            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # 4(w), 4(h), 128(out_channels)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256), # 256 is a random value for neutorns
            nn.ReLU(),

            nn.Linear(256, 10) # 10 classes
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # flattern
        return self.fc_layers(x)


In [29]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Total NVIDIA GPUs PyTorch can see:", torch.cuda.device_count())
print("Name of GPU 0:", torch.cuda.get_device_name(0))
print(f"Currently using: {torch.cuda.memory_allocated(device)} GB")

model = CNN().to(device=device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

Total NVIDIA GPUs PyTorch can see: 1
Name of GPU 0: NVIDIA GeForce MX350
Currently using: 27761152 GB


In [30]:
epochs = 10

for epoch in range(0, epochs):
    training_loss = 0.0

    for images, lables in trainloader:
        images = images.to(device)
        lables = lables.to(device)
        
        optimizer.zero_grad()
        outputs = model.forward(images)
        loss = criterion(outputs, lables)
        training_loss += loss.item()
        
        loss.backward()
        optimizer.step()

    model.eval()
    correct_labels = 0
    total_lables = 0
    with torch.no_grad():
        for images, lables in testloader:
            images = images.to(device)
            lables = lables.to(device)

            outputs = model.forward(images)
            _, predicted = torch.max(outputs, 1)

            correct_labels = (predicted == lables).sum().item()
            total_lables = lables.size(0)

            


    print(f" Epoch = {epoch + 1}/{epochs} | Loss = {training_loss/ len(trainloader)}  | Valid Acc {correct_labels / total_lables * 100}")

 Epoch = 1/10 | Loss = 1.3580658014320657  | Valid Acc 75.0
 Epoch = 2/10 | Loss = 0.9121103092380192  | Valid Acc 56.25
 Epoch = 3/10 | Loss = 0.7341902163403723  | Valid Acc 75.0
 Epoch = 4/10 | Loss = 0.6147371591509455  | Valid Acc 81.25
 Epoch = 5/10 | Loss = 0.5128554306790957  | Valid Acc 81.25
 Epoch = 6/10 | Loss = 0.4212203930939555  | Valid Acc 75.0
 Epoch = 7/10 | Loss = 0.3430824260063031  | Valid Acc 81.25
 Epoch = 8/10 | Loss = 0.26774269171878506  | Valid Acc 81.25
 Epoch = 9/10 | Loss = 0.2087802931504405  | Valid Acc 81.25
 Epoch = 10/10 | Loss = 0.1702113887604774  | Valid Acc 81.25


In [32]:
correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, lables in testloader:
        images = images.to(device)
        lables = lables.to(device)
        
        outpus = model.forward(images)
        _,predicted = torch.max(outpus, 1)

        correct_labels += (predicted == lables).sum().item()
        total_labels += lables.size(0)

print(f"Accuracy: {correct_labels / total_labels * 100} ")

Accuracy: 75.68 
